# Fase 1: Arquitectura de Conocimiento (RAG)
### Tutor Analítico Híbrido (RAG + Fine-Tuning)

Este cuaderno implementa el pipeline completo de procesamiento del corpus documental, chunking, generación de embeddings y almacenamiento en la base de datos vectorial local **ChromaDB**. Esto constituye la "memoria a largo plazo" del tutor.

In [1]:
# 1. Instalar dependencias necesarias
!pip install pymupdf sentence-transformers chromadb openpyxl pandas -q

### 2. Procesamiento del Corpus Documental
Extraemos texto de los documentos del corpus en `data/corpus/`. Procesamos tanto archivos **PDF** (usando `fitz`/PyMuPDF para registrar la página exacta de donde proviene el texto) como archivos **XLSX** (usando `pandas` para estructurar los datos de las hojas de cálculo como pasajes legibles de texto).

In [2]:
import os
import glob
import fitz  # PyMuPDF
import pandas as pd

corpus_dir = os.path.join('data', 'corpus')

def extract_text_from_pdf(filepath):
    """Extrae texto por páginas manteniendo el número de página como metadata."""
    documents = []
    filename = os.path.basename(filepath)
    try:
        doc = fitz.open(filepath)
        for page_num in range(len(doc)):
            page = doc[page_num]
            text = page.get_text().strip()
            if text:
                documents.append({
                    'text': text,
                    'metadata': {
                        'source': filename,
                        'page': page_num + 1,
                        'type': 'pdf'
                    }
                })
    except Exception as e:
        print(f"Error al leer el PDF {filename}: {e}")
    return documents

def extract_text_from_xlsx(filepath):
    """Convierte el contenido de archivos Excel a descripciones de datos tabulares."""
    documents = []
    filename = os.path.basename(filepath)
    try:
        xl = pd.ExcelFile(filepath)
        for sheet_name in xl.sheet_names:
            df = xl.parse(sheet_name)
            # Convertir filas de la tabla a representación de texto
            rows_text = []
            # Tomar las primeras 100 filas máximo para evitar saturación de contexto
            for idx, row in df.head(100).iterrows():
                row_desc = ", ".join([f"{col}: {val}" for col, val in row.items() if pd.notna(val)])
                rows_text.append(f"Registro {idx+1}: {row_desc}")
            
            if rows_text:
                full_sheet_text = f"Hoja: {sheet_name} en archivo {filename}. Datos:\n" + "\n".join(rows_text)
                documents.append({
                    'text': full_sheet_text,
                    'metadata': {
                        'source': filename,
                        'page': sheet_name,
                        'type': 'xlsx'
                    }
                })
    except Exception as e:
        print(f"Error al leer el Excel {filename}: {e}")
    return documents

### 3. Segmentación del Texto (Chunking)
Dividimos los textos extraídos en fragmentos (*chunks*) más pequeños. Esto es indispensable para optimizar la ventana de contexto de los modelos de lenguaje (LLM) y asegurar búsquedas semánticas precisas. Definimos un `chunk_size` de ~500 caracteres con un `overlap` de ~100 caracteres.

In [ ]:
def split_text_into_chunks(documents, chunk_size=500, overlap=100):
    """Divide el texto de los documentos en chunks pequeños con solape."""
    chunks = []
    for doc in documents:
        text = doc['text']
        metadata = doc['metadata']
        
        
        if len(text) <= chunk_size:
            chunks.append({
                'text': text,
                'metadata': metadata
            })
            continue
            
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]
            
            
            chunk_meta = metadata.copy()
            chunk_meta['chunk_start'] = start
            
            chunks.append({
                'text': chunk_text,
                'metadata': chunk_meta
            })
            start += (chunk_size - overlap)
            
    return chunks

### 4. Indexación en Base de Datos Vectorial (ChromaDB)
Inicializamos ChromaDB de forma local y persistente en disco. Usaremos el modelo de embeddings multilingüe `paraphrase-multilingual-MiniLM-L12-v2` de la librería `sentence-transformers`, ideal para indexar y buscar información técnica en idioma español.

In [4]:
import chromadb
from chromadb.utils import embedding_functions

# Inicializar el cliente persistente local de ChromaDB
chroma_client = chromadb.PersistentClient(path="data/vectorstore")

# Cargar la función de embeddings multilingüe en español
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Crear o recuperar la colección de documentos de seguridad
collection = chroma_client.get_or_create_collection(
    name="tutor_seguridad_corpus",
    embedding_function=embedding_func
)

c:\Users\crist\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\crist\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\crist\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate De

### 5. Ejecutar la Ingesta de Datos en Lote
Leemos todos los archivos del corpus en `data/corpus/`, extraemos el texto, creamos los chunks y los subimos a ChromaDB asociándoles IDs únicos y su correspondiente metadata.

In [5]:
raw_docs = []
pdf_files = glob.glob(os.path.join(corpus_dir, "*.pdf"))
xlsx_files = glob.glob(os.path.join(corpus_dir, "*.xlsx"))

print(f"Encontrados {len(pdf_files)} PDFs y {len(xlsx_files)} Excels.")

# Extraer textos
for f in pdf_files:
    print(f"Procesando PDF: {os.path.basename(f)}")
    raw_docs.extend(extract_text_from_pdf(f))

for f in xlsx_files:
    print(f"Procesando Excel: {os.path.basename(f)}")
    raw_docs.extend(extract_text_from_xlsx(f))

# Segmentar en chunks
all_chunks = split_text_into_chunks(raw_docs, chunk_size=500, overlap=100)
print(f"Total de chunks generados: {len(all_chunks)}")

# Ingestar en ChromaDB
ids = [f"id_{i}" for i in range(len(all_chunks))]
documents_content = [chunk['text'] for chunk in all_chunks]
# Convertimos la metadata a tipos soportados por ChromaDB (cadenas, enteros, flotantes)
metadatas = []
for chunk in all_chunks:
    m = {}
    for k, v in chunk['metadata'].items():
        m[k] = str(v) if not isinstance(v, (int, float, bool)) else v
    metadatas.append(m)

# Subir por bloques para evitar saturación de memoria
batch_size = 500
for i in range(0, len(all_chunks), batch_size):
    collection.add(
        ids=ids[i:i+batch_size],
        documents=documents_content[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size]
    )

print("¡Ingesta completada con éxito en ChromaDB!")

Encontrados 19 PDFs y 3 Excels.
Procesando PDF: Abandono-escolar-en-Educacion-Basica-2019-2023_.pdf
Procesando PDF: asun_5085425_20260429_1772049997.pdf
Procesando PDF: Atrocidades 2025.pdf
Procesando PDF: CNSP-Unidades_robadas_2015-2026_abr26.pdf
Procesando PDF: Delitos-100_mil_hab_2015-2026_abr26.pdf
Procesando PDF: enape_2021_nota_tecnica.pdf
Procesando PDF: Encuesta Nacional sobre Acceso y Permanencia en la Educacion.pdf
Procesando PDF: ESTUDIO-RECLUTADOS-POR-LA-DELINCUENCIA-ORGANIZADA.pdf
Procesando PDF: Fuerofederal_abr_2026.pdf
Procesando PDF: Informe_IncidenciaDelictiva_Fuero_Comun_Abril_2026.pdf
Procesando PDF: Manual_metodol_gico_RNID_V1.0_VF.pdf
Procesando PDF: page-one-noviembre.pdf
Procesando PDF: pdfcandle.com-Info-delict-violencia_contra_las_mujeres_Abr26_compressed.pdf
Procesando PDF: principales_cifras_2024_2025_bolsillo.pdf
Procesando PDF: RNID-Delitos-2026_abr26.pdf
Procesando PDF: RNID-V_ctimas-2026_abr26.pdf
Procesando PDF: secuestrofederal_abr_2026.pdf
Procesando 

### 6. Sistema de Búsqueda Semántica (Retriever)
Probamos el retriever semántico (Top-K) que inyectará el contexto a nuestro Tutor durante las consultas.

In [6]:
def retrieve_context(query, top_k=5):
    """Realiza una búsqueda semántica de los chunks más similares."""
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    
    retrieved_chunks = []
    if results and results['documents']:
        for i in range(len(results['documents'][0])):
            doc_text = results['documents'][0][i]
            meta = results['metadatas'][0][i]
            retrieved_chunks.append({
                'text': doc_text,
                'source': meta.get('source', 'Desconocido'),
                'page': meta.get('page', 'N/A')
            })
    return retrieved_chunks

# Prueba rápida de recuperación semántica
test_query = "¿Cuáles son los tres estados con mayor índice de homicidios dolosos?"
context = retrieve_context(test_query, top_k=3)

print(f"Consulta: {test_query}\n")
for idx, chunk in enumerate(context, 1):
    print(f"[Fragmento {idx}] de: {chunk['source']} (Pág/Hoja {chunk['page']})")
    print(f"Contenido: {chunk['text']}\n" + "-"*50)

Consulta: ¿Cuáles son los tres estados con mayor índice de homicidios dolosos?

[Fragmento 1] de: page-one-noviembre.pdf (Pág/Hoja 1)
Contenido: nternacionales. 
En este contexto, la pacificación del territorio continúa sien-
do un desafío. Pese a la reducción nacional en el registro de 
homicidios dolosos, 16 estados siguen registrando condicio-
nes preocupantes de violencia letal.
En varios de estos estados, los episodios de violencia coinci-
den con intervenciones federales enfocadas en debilitar ope-
rativamente a las organizaciones criminales. Sin embargo, 
cuando estas acciones no están acompañadas por capacida-
des institucionales
--------------------------------------------------
[Fragmento 2] de: vap-anual-dic-2025.pdf (Pág/Hoja 16)
Contenido:  el comportamiento es 
igualmente preocupante: en un estado 
que concentra una parte sustantiva del 
homicidio doloso nacional, la expansión 
de esta categoría apunta a que parte de 
la violencia letal o cuasi letal podría estar 
desplaz